<a href="https://colab.research.google.com/github/tsgebre/Flood_Physics_Guided_DL/blob/main/experiment_ems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Agent Role: Experiment Evidence & Visualization Engineer**

You are an expert ML research engineer specializing in **experiment reproducibility, evaluation pipelines, and publication-quality visualization**.

Your mission is to **translate evidence requirements (figures, tables, comparisons) into stand-alone scripts** that generate the required results and visuals from existing experiment artifacts.

The notebook contains:

* experiment scripts
* training logs
* example metrics

However:

* **experiments and checkpoints exist on a local machine**
* the scripts you propose will be **executed locally**

Your responsibility is to **design the minimal scripts required to generate the evidences recommended by the previous agent**.

Do **not redesign experiments** unless strictly necessary.

---

# **Primary Objectives**

1. Map evidence requirements to reproducible evaluation workflows.
2. Design **stand-alone scripts** that:

   * load checkpoints
   * run inference or evaluation
   * compute metrics
   * export results
3. Generate **publication-ready figures and tables**.
4. Ensure outputs are easily reproducible and compatible with journal-quality presentation.

---

# **Expected Inputs**

The agent may receive:

* Evidence plan from the previous agent
* Notebook containing experiment scripts and metrics
* Dataset descriptions or loaders
* Partial results or logs
* Model checkpoints (stored locally)

---

# **Input Validation**

Before analysis verify the availability of:

* experiment scripts
* checkpoint paths
* dataset loaders
* metric definitions

If critical components are missing, list them and suggest minimal additions required.

---

# **Operational Framework**

## Phase 1 — Evidence Mapping

For each evidence item determine:

* dataset required
* checkpoint required
* evaluation metric
* expected output artifact

Identify whether the evidence requires:

* inference runs
* metric computation
* aggregation of results
* visualization generation

---

## Phase 2 — Execution Script Design

Design **stand-alone scripts** that can be executed locally.

Scripts may include:

* inference runners
* evaluation pipelines
* metric aggregation utilities
* visualization generators

Each script should specify:

* required inputs
* expected outputs
* execution steps

Prefer **minimal additions** to existing code.

Avoid retraining models unless necessary.

---

## Phase 3 — Result Logging & Artifact Structure

Define reproducible outputs using structured formats:

Recommended artifacts:

```
metrics.csv
results_summary.csv
predictions.npy
metadata.json
```

Each artifact should record:

* experiment_id
* checkpoint_used
* dataset_split
* metric_definition

---

## Phase 4 — Publication-Quality Visualization Design

For each required visual element, specify:

* figure type (plot, chart, table)
* data source
* variables shown
* layout structure

Ensure figures are **publication-ready**:

Guidelines:

* font sizes appropriate for journal figures
* clear axis labeling and legends
* colorblind-safe color palettes
* consistent styling across figures
* export formats suitable for papers (`PDF`, `SVG`, high-resolution `PNG`)

Tables should be structured for **direct inclusion in manuscripts**.

---

# **Output Format**

### 1. Evidence Implementation Summary

Mapping between evidence items and required experiment outputs.

### 2. Missing Components

List scripts or utilities needed to generate results.

### 3. Proposed Execution Scripts

For each script describe:

* purpose
* inputs
* outputs
* execution steps

(No full code.)

### 4. Experiment Output Structure

Example:

```
artifacts/
   experiment_01/
      metrics.csv
      predictions.npy
      metadata.json
```

### 5. Visualization Specification

Describe the figures/tables to be generated and how they should present the results.

### 6. Evidence Readiness Check

Confirm whether the outputs will support the required figures and tables.

---

# **Escalation Condition**

Escalate as **“Evidence Generation Blocked”** if:

* checkpoints are unavailable
* datasets cannot be accessed
* required metrics cannot be computed

Provide minimal corrective actions.

---

# **Behavioral Constraints**

* Prefer **evaluation and inference using existing checkpoints**.
* Avoid generic ML advice.
* Focus strictly on **scripts and workflows needed to generate evidence**.
* Ensure figures are **clear, consistent, and publication-ready**.

---

# **Tone**

Technical, concise, and implementation-focused.
Assume an **ML research audience preparing reproducible experiments for publication**.

# models

```
# src/models.py
# Ordered by experimental priority:
# 1) MLP → 2) RNN → 3) TCN → 4) LSTM → 5) GRU → 6) Transformer

import torch
import torch.nn as nn
import torch.nn.functional as F


# =========================================================
# 1. MLP (Primary Hypothesis Test)
# =========================================================
class PhysicsInformedMLP(nn.Module):
    """
    Critical control:
    Tests whether physics helps WITHOUT temporal modeling.
    If yes → physics signal is architecture-agnostic.
    """
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim,
                 init_rain, init_flood, init_weight=0.0):
        super().__init__()

        layers = []
        in_dim = input_dim
        for _ in range(layer_dim):
            layers.append(nn.Linear(in_dim, hidden_dim))
            layers.append(nn.ReLU())
            in_dim = hidden_dim

        self.mlp = nn.Sequential(*layers)
        self.fc = nn.Linear(hidden_dim, output_dim)

        # Physics params
        self.w_monotonicity = nn.Parameter(torch.tensor(init_weight, dtype=torch.float32))
        self.w_threshold    = nn.Parameter(torch.tensor(init_weight, dtype=torch.float32))
        self.thresh_rain    = nn.Parameter(torch.tensor(init_rain, dtype=torch.float32))
        self.thresh_flood   = nn.Parameter(torch.tensor(init_flood, dtype=torch.float32))

    def forward(self, x):
        # x: [B, T, F]

        B, T, F = x.shape

        # collapse time into batch (strict independence)
        x = x.reshape(B * T, F)

        out = self.mlp(x)
        out = self.fc(out)

        # reshape back
        out = out.reshape(B, T, -1)

        return out


# =========================================================
# 2. RNN (Weak Learner Test)
# =========================================================
class PhysicsInformedRNN(nn.Module):
    """
    Simplest recurrent model.
    Expect strong gains from physics.
    """
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim,
                 init_rain, init_flood, init_weight=0.0):
        super().__init__()

        self.rnn = nn.RNN(
            input_dim, hidden_dim, layer_dim,
            batch_first=True, nonlinearity='tanh'
        )
        self.fc = nn.Linear(hidden_dim, output_dim)

        # Physics params
        self.w_monotonicity = nn.Parameter(torch.tensor(init_weight, dtype=torch.float32))
        self.w_threshold    = nn.Parameter(torch.tensor(init_weight, dtype=torch.float32))
        self.thresh_rain    = nn.Parameter(torch.tensor(init_rain, dtype=torch.float32))
        self.thresh_flood   = nn.Parameter(torch.tensor(init_flood, dtype=torch.float32))

        self.layer_dim  = layer_dim
        self.hidden_dim = hidden_dim

    def forward(self, x):
        h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(x.device)
        out, _ = self.rnn(x, h0)
        return self.fc(out)


# =========================================================
# 3. TCN (Non-Recurrent Temporal Baseline)
# =========================================================
class PhysicsInformedTCN(nn.Module):
    """
    Removes recurrence entirely.
    Tests temporal modeling via convolutions.
    Often strong in hydrology tasks.
    """
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim,
                 init_rain, init_flood, init_weight=0.0, kernel_size=3):
        super().__init__()

        layers = []
        in_channels = input_dim

        for i in range(layer_dim):
            layers.append(
                nn.Conv1d(
                    in_channels,
                    hidden_dim,
                    kernel_size,
                    padding=0,
                    dilation=2**i
                )
            )
            layers.append(nn.ReLU())
            in_channels = hidden_dim

        self.tcn = nn.Sequential(*layers)
        self.fc  = nn.Linear(hidden_dim, output_dim)

        # Physics params
        self.w_monotonicity = nn.Parameter(torch.tensor(init_weight, dtype=torch.float32))
        self.w_threshold    = nn.Parameter(torch.tensor(init_weight, dtype=torch.float32))
        self.thresh_rain    = nn.Parameter(torch.tensor(init_rain, dtype=torch.float32))
        self.thresh_flood   = nn.Parameter(torch.tensor(init_flood, dtype=torch.float32))
    
    def forward(self, x):
        # x: [B, T, F]
        x = x.transpose(1, 2)  # [B, F, T]

        for layer in self.tcn:
            if isinstance(layer, nn.Conv1d):
                k = layer.kernel_size[0]
                d = layer.dilation[0]

                # causal left padding only
                pad = (k - 1) * d
                x = F.pad(x, (pad, 0))  # ONLY past context

                x = layer(x)
            else:
                x = layer(x)

        x = x.transpose(1, 2)
        return self.fc(x)

# =========================================================
# 4. LSTM (Stable Recurrent Baseline)
# =========================================================
class PhysicsInformedLSTM(nn.Module):
    """
    Standard gated recurrent model.
    Expect moderate improvement from physics.
    """
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim,
                 init_rain, init_flood, init_weight=0.0):
        super().__init__()

        self.lstm = nn.LSTM(input_dim, hidden_dim, layer_dim, batch_first=True)
        self.fc   = nn.Linear(hidden_dim, output_dim)

        # Physics params
        self.w_monotonicity = nn.Parameter(torch.tensor(init_weight, dtype=torch.float32))
        self.w_threshold    = nn.Parameter(torch.tensor(init_weight, dtype=torch.float32))
        self.thresh_rain    = nn.Parameter(torch.tensor(init_rain, dtype=torch.float32))
        self.thresh_flood   = nn.Parameter(torch.tensor(init_flood, dtype=torch.float32))

        self.layer_dim  = layer_dim
        self.hidden_dim = hidden_dim

    def forward(self, x):
        h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(x.device)
        c0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(x.device)

        out, _ = self.lstm(x, (h0, c0))
        return self.fc(out)


# =========================================================
# 5. GRU (Efficiency vs Stability)
# =========================================================
class PhysicsInformedGRU(nn.Module):
    """
    Simplified gating vs LSTM.
    Expect mixed / unstable results.
    """
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim,
                 init_rain, init_flood, init_weight=0.0):
        super().__init__()

        self.gru = nn.GRU(input_dim, hidden_dim, layer_dim, batch_first=True)
        self.fc  = nn.Linear(hidden_dim, output_dim)

        # Physics params
        self.w_monotonicity = nn.Parameter(torch.tensor(init_weight, dtype=torch.float32))
        self.w_threshold    = nn.Parameter(torch.tensor(init_weight, dtype=torch.float32))
        self.thresh_rain    = nn.Parameter(torch.tensor(init_rain, dtype=torch.float32))
        self.thresh_flood   = nn.Parameter(torch.tensor(init_flood, dtype=torch.float32))

        self.layer_dim  = layer_dim
        self.hidden_dim = hidden_dim

    def forward(self, x):
        h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(x.device)
        out, _ = self.gru(x, h0)
        return self.fc(out)


# =========================================================
# 6. Transformer (Attention-Based Test)
# =========================================================
import math
import torch

class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe.unsqueeze(0))  # [1, T, D]

    def forward(self, x):
        # x: [B, T, D]
        T = x.size(1)
        return x + self.pe[:, :T]

class PhysicsInformedTransformer(nn.Module):
    """
    Lightweight attention model.
    Keep small for fair comparison.
    Outcome: exploratory / unclear.
    """
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim,
                 init_rain, init_flood, init_weight=0.0, nhead=2):
        super().__init__()

        self.input_proj = nn.Linear(input_dim, hidden_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,            # match hidden_dim
            nhead=nhead,
            dim_feedforward=hidden_dim*2,  # also match
            batch_first=True
        )
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.pos_enc = SinusoidalPositionalEncoding(hidden_dim)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=layer_dim)
        self.fc = nn.Linear(hidden_dim, output_dim)

        # Physics params
        self.w_monotonicity = nn.Parameter(torch.tensor(init_weight, dtype=torch.float32))
        self.w_threshold    = nn.Parameter(torch.tensor(init_weight, dtype=torch.float32))
        self.thresh_rain    = nn.Parameter(torch.tensor(init_rain, dtype=torch.float32))
        self.thresh_flood   = nn.Parameter(torch.tensor(init_flood, dtype=torch.float32))
   

    def forward(self, x):
        # x: [B, T, F]

        x = self.input_proj(x)
        x = self.pos_enc(x)

        out = self.transformer(x)
        return self.fc(out)
```



# original



```
# run_uq_tuned6_v11_fixed.py

import torch
import numpy as np
from torch.utils.data import DataLoader
import random
import os
import sys
import copy
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import logging

sys.path.append(os.path.join(os.path.dirname(__file__), 'src'))
import src.utils as utils
import src.models as models
import src.dataset as dataset

# -----------------------------
# Logging
# -----------------------------
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

# -----------------------------
# Seed + Determinism
# -----------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# -----------------------------
# Hydro Prior (fixed usage)
# -----------------------------
class HydroPrior(nn.Module):
    def __init__(self, L):
        super().__init__()
        self.kernel_raw = nn.Parameter(torch.ones(L))
        self.L = L

    def forward(self, r):
        kernel = torch.softmax(self.kernel_raw, dim=0)
        Q = F.conv1d(
            r.transpose(1, 2),
            kernel.view(1, 1, self.L),
            padding=self.L - 1
        )
        return Q[:, 0, :r.size(1)]

# -----------------------------
# Mass balance loss (new)
# -----------------------------

def mass_balance_loss(q, r, extreme, runoff_coeff=0.8, eps=1e-6):
    """
    Event-based mass balance constraint:
    sum(q) <= c * sum(r)

    q: [B, T, 1]
    r: [B, T, 1]
    extreme: rainfall mask
    """

    # mask rainfall periods
    q_masked = q * extreme
    r_masked = r * extreme

    sum_q = q_masked.sum(dim=1)
    sum_r = r_masked.sum(dim=1)

    # upper bound (physical constraint)
    upper_violation = F.relu(sum_q - runoff_coeff * sum_r)

    # optional lower bound (weak, prevents trivial zero flow)
    lower_violation = F.relu(0.1 * sum_r - sum_q)

    # normalize (IMPORTANT)
    norm = sum_r + eps

    loss = (upper_violation / norm) + 0.3 * (lower_violation / norm)

    return loss.mean()

# =========================
# IMPULSE-RESPONSE FORCING
# =========================
def compute_effective_rainfall(r, decay=0.8):
    """
    r: (B, T, 1)
    returns: (B, T, 1) smoothed / lag-aware rainfall
    """
    B, T, _ = r.shape
    S = torch.zeros_like(r)

    for t in range(T):
        if t == 0:
            S[:, t, :] = r[:, t, :]
        else:
            S[:, t, :] = decay * S[:, t-1, :] + r[:, t, :]

    return S

# -----------------------------
# TRAIN (fixed minimal changes)
# -----------------------------
def train_model_v10_adaptive(
    model,
    train_loader,
    val_loader,
    lr,
    use_physics=False,
    use_hydro_prior=False,
    hydro_prior=None,
    use_mass_balance=False,
    epochs=100,
    patience=10,
    warmup_epochs=10,
    device='cpu',
    lambda_mono=0.3,
    lambda_thresh=0.3,
    lambda_hydro=0.1,
    mono_horizon=3,
    lambda_mass=0.05
):

    optimizer = optim.AdamW(model.parameters(), lr=lr)

    if use_hydro_prior and hydro_prior is not None:
        hydro_prior.eval()

    best_val_loss = float('inf')
    best_weights = copy.deepcopy(model.state_dict())
    patience_counter = 0

    for epoch in range(epochs):

        model.train()

        # smoother warmup (avoid sharp physics jump)
        if use_physics:
            alpha = min(1.0, (epoch / warmup_epochs))
        else:
            alpha = 0.0

        for batch in train_loader:
            x = batch['x'].to(device)
            y = batch['y'].to(device)

            optimizer.zero_grad()

            q = model(x)

            # expand target across time dimension
            y_seq = y.unsqueeze(1).expand(-1, q.shape[1], -1)

            # sequence supervision loss
            loss_data = F.mse_loss(q, y_seq)

            loss_phys = torch.tensor(0.0, device=device)

            # =========================
            # PHYSICS LOSS
            # =========================
            if use_physics:
                r = x[:, :, 0].unsqueeze(-1)

                t_rain = model.thresh_rain
                t_flood = model.thresh_flood

                extreme = (r > t_rain).float()

                # ---- monotonicity (impulse + derivative) ----

                # effective rainfall (storage-like smoothing)

                r_eff = compute_effective_rainfall(r, decay=0.8)

                # normalize for stability
                r_eff = r_eff / (r_eff.mean(dim=1, keepdim=True) + 1e-6)

                # ---- align sequence lengths safely ----
                T = min(q.shape[1], r_eff.shape[1])

                q_c = q[:, :T, :]
                r_eff_c = r_eff[:, :T, :]

                # compute derivative AFTER alignment
                forcing = r_eff_c[:, 1:, :]

                # enforce rainfall-response consistency instead of dq dynamics
                q_next = q_c[:, 1:, :]
                q_prev = q_c[:, :-1, :]

                r_influence = forcing  # already aligned

                # softer constraint: response should not drop under sustained forcing
                loss_mono = (F.relu(q_prev - q_next) * r_influence).mean()

                # ---- threshold constraint ----
                t_dyn = t_flood + 0.1 * q[:, :-1, :]

                # align sequence lengths dynamically
                T = min(t_dyn.shape[1], q.shape[1] - 1, extreme.shape[1])
                t_dyn_aligned = t_dyn[:, :T, :]
                q_aligned = q[:, :T, :]
                extreme_aligned = extreme[:, :T, :]

                loss_thresh = (F.relu(t_dyn_aligned - q_aligned) * extreme_aligned).mean()

                loss_phys = (
                    lambda_mono * loss_mono +
                    lambda_thresh * loss_thresh
                )

                # =========================
                # HYDRO PRIOR
                # =========================
                if use_hydro_prior and hydro_prior is not None:
                    Q_prior = hydro_prior(r).unsqueeze(-1).detach()

                    # shape safety
                    T = min(q.shape[1], Q_prior.shape[1], extreme.shape[1])
                    q_c = q[:, :T, :]
                    p_c = Q_prior[:, :T, :]
                    m_c = extreme[:, :T, :]

                    # centered comparison
                    q_c = q_c - q_c.mean(dim=1, keepdim=True)
                    p_c = p_c - p_c.mean(dim=1, keepdim=True)

                    diff2 = (q_c - p_c) ** 2
                    mask_sum = m_c.sum() + 1e-6
                    loss_hydro = (diff2 * m_c).sum() / mask_sum

                    # delayed activation
                    hydro_gate = min(1.0, max(0.0, (epoch - 5) / warmup_epochs))

                    loss_phys = loss_phys + lambda_hydro * hydro_gate * loss_hydro

                # =========================
                # MASS BALANCE
                # =========================
                if use_mass_balance:
                    loss_mass = mass_balance_loss(q, r, extreme)

                    # delayed activation
                    mass_gate = min(1.0, max(0.0, (epoch - 3) / warmup_epochs))

                    loss_phys = loss_phys + lambda_mass * mass_gate * loss_mass

            # =========================
            # TOTAL LOSS
            # =========================
            loss = loss_data + alpha * loss_phys

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        # =========================
        # VALIDATION
        # =========================
        model.eval()
        val_losses = []

        with torch.no_grad():
            for batch in val_loader:
                x = batch['x'].to(device)
                y = batch['y'].to(device)

                pred = model(x)[:, -1, :]
                val_losses.append(F.mse_loss(pred, y).item())

        val_loss = np.mean(val_losses)

        logging.info(
            f"Epoch {epoch+1} | Val Loss: {val_loss:.4f} | Alpha: {alpha:.3f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    model.load_state_dict(best_weights)
    return model

# =========================================================
# LOGGING (MUST BE FIRST)
# =========================================================
os.makedirs("results", exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler("results/run.log"),
        logging.StreamHandler()
    ]
)

# =========================================================
# CONFIGURATIONS (FROM OPTUNA, FIXED)
# =========================================================
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def match_hidden_dim(model_class, cfg, init_rain, init_flood, target_params, device,
                     h_min=16, h_max=1024, tol=0.05, model_type=None):

    best_h = h_min
    best_diff = float("inf")

    nhead = 2 if model_type == "Transformer" else None

    # ensure start is valid
    def make_valid(h):
        if nhead is not None:
            return max(nhead, (h // nhead) * nhead)
        return h

    h_min = make_valid(h_min)
    h_max = make_valid(h_max)

    while h_min <= h_max:
        h_mid = (h_min + h_max) // 2
        h_mid = make_valid(h_mid)

        if h_mid < nhead if nhead else False:
            h_mid = nhead

        try:
            model = model_class(
                input_dim=2,
                hidden_dim=h_mid,
                layer_dim=cfg["layer_dim"],
                output_dim=1,
                init_rain=init_rain,
                init_flood=init_flood,
                init_weight=0.0
            ).to(device)

            n_params = count_parameters(model)

        except Exception:
            # skip invalid configs safely
            h_min = h_mid + 1
            continue

        diff = abs(n_params - target_params) / target_params

        if diff < best_diff:
            best_diff = diff
            best_h = h_mid

        if n_params > target_params:
            h_max = h_mid - nhead if nhead else h_mid - 1
        else:
            h_min = h_mid + nhead if nhead else h_mid + 1

        if diff < tol:
            break

    return best_h

def get_target_params(cfg, init_rain, init_flood, device, reference="LSTM"):

    ref_map = {
        "LSTM": models.PhysicsInformedLSTM,
        "GRU": models.PhysicsInformedGRU
    }

    ref_model = ref_map[reference](
        input_dim=2,
        hidden_dim=cfg["hidden_dim"],
        layer_dim=cfg["layer_dim"],
        output_dim=1,
        init_rain=init_rain,
        init_flood=init_flood,
        init_weight=0.0
    ).to(device)

    target = count_parameters(ref_model)
    logging.info(f"[Reference: {reference}] params = {target}")

    return target

# =========================================================
# MODEL BUILDING
# =========================================================
def build_model_matched(model_type, cfg, init_rain, init_flood, device, target_params):

    model_map = {
        "MLP": models.PhysicsInformedMLP,
        "RNN": models.PhysicsInformedRNN,
        "TCN": models.PhysicsInformedTCN,
        "LSTM": models.PhysicsInformedLSTM,
        "GRU": models.PhysicsInformedGRU,
        "Transformer": models.PhysicsInformedTransformer,
    }

    model_class = model_map[model_type]

    # LSTM/GRU = reference → keep original hidden_dim
    if model_type in ["LSTM", "GRU"]:
        hidden_dim = cfg["hidden_dim"]
    else:
        hidden_dim = match_hidden_dim(
            model_class,
            cfg,
            init_rain,
            init_flood,
            target_params,
            device,
            model_type=model_type   # <-- ADD THIS
        )

    model = model_class(
        input_dim=2,
        hidden_dim=hidden_dim,
        layer_dim=cfg["layer_dim"],
        output_dim=1,
        init_rain=init_rain,
        init_flood=init_flood,
        init_weight=0.0
    ).to(device)

    logging.info(
        f"{model_type} | hidden_dim={hidden_dim} | params={count_parameters(model)}"
    )

    return model
# =========================================================
# RUN EXPERIMENT (FIXED SIGNATURE)
# =========================================================
def run_experiment(cfg, model_type, use_physics, device, dataframes, use_hydro_prior=False, use_mass_balance=False):

    set_seed(42)

    init_rain, init_flood = dataframes["thresholds"]

    train_loader = DataLoader(
        dataset.CamelsDataset(dataframes["train"], cfg["seq_length"], True),
        batch_size=cfg["batch_size"],
        shuffle=True,
        num_workers=0  # <- critical for determinism
    )
    val_loader = DataLoader(
        dataset.CamelsDataset(dataframes["val"], cfg["seq_length"], False),
        batch_size=cfg["batch_size"],
        num_workers=0
    )
    test_loader = DataLoader(
        dataset.CamelsDataset(dataframes["test"], cfg["seq_length"], False),
        batch_size=cfg["batch_size"],
        num_workers=0
    )

    # compute once per config (outside loop ideally)
    target_params = get_target_params(cfg, init_rain, init_flood, device, reference="LSTM")

    model = build_model_matched(
        model_type,
        cfg,
        init_rain,
        init_flood,
        device,
        target_params
    )

    hydro_prior = None  # removed (not used)

    model = train_model_v10_adaptive(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        lr=cfg["lr"],
        use_physics=use_physics,
        use_hydro_prior=False,
        hydro_prior=hydro_prior,
        epochs=100,
        patience=12,
        warmup_epochs=7,
        device=device,
        lambda_mono=cfg.get("lambda_mono", 0.5),
        lambda_thresh=cfg.get("lambda_thresh", 0.5),
        mono_horizon=cfg.get("mono_horizon", 3)
    )

    return model, test_loader

# =========================================================
# EVALUATION FUNCTION
# =========================================================
def evaluate(model, loader, device, scaler_d, level):

    preds, targets = utils.evaluate_stress_test(
        model, loader, scaler_d,
        mask_ratio=level,
        device=device
    )

    nse = utils.calculate_nse(targets, preds)

    thr = np.percentile(targets, 90)
    mask = targets > thr
    mae = np.mean(np.abs(preds[mask] - targets[mask])) if np.any(mask) else 0.0

    return nse, mae

# =========================================================
# CONFIG LOOP
# =========================================================
SCARCITY_LEVELS = [0.0, 0.2, 0.5, 0.8]
M_BOOTSTRAPS = 30

def evaluate_configurations(
    model_type,
    use_physics,
    device,
    dataframes,
    scaler_d,
    configs
):

    results = {}

    for i, cfg in enumerate(configs):

        logging.info(
            f"\n===== Config {i+1} | {model_type} | Physics={use_physics} ====="
        )

        model, test_loader = run_experiment(
            cfg=cfg,
            model_type=model_type,
            use_physics=use_physics,
            device=device,
            dataframes=dataframes
        )

        results[f"Config_{i+1}"] = {}

        for level in SCARCITY_LEVELS:

            runs = 1 if level == 0 else M_BOOTSTRAPS
            nses, maes = [], []

            for _ in range(runs):
                nse, mae = evaluate(model, test_loader, device, scaler_d, level)
                nses.append(nse)
                maes.append(mae)

            results[f"Config_{i+1}"][level] = {
                "nse_mean": np.mean(nses),
                "nse_std": np.std(nses),
                "mae_mean": np.mean(maes),
                "mae_std": np.std(maes),
            }

    return results

# =========================================================
# PRINTING (UNCHANGED STYLE)
# =========================================================
def print_results(results, title):

    logging.info(f"\n========== {title} ==========")

    for cfg, levels in results.items():

        logging.info(f"\n🔧 {cfg}")

        for level in sorted(levels.keys()):

            m = levels[level]

            logging.info(
                f"{int(level*100):>3}% | "
                f"NSE {m['nse_mean']:.3f} ± {m['nse_std']:.3f} | "
                f"MAE {m['mae_mean']:.2f} ± {m['mae_std']:.2f}"
            )

# =========================================================
# COMPARISON (YOUR ORIGINAL NICE FORMAT KEPT)
# =========================================================
def compare_results(res_a, res_b, title_a="A", title_b="B"):

    logging.info(f"\n========== COMPARISON: {title_a} vs {title_b} ==========")

    for cfg in res_a.keys():

        logging.info(f"\n🔧 {cfg}")

        for level in sorted(res_a[cfg].keys()):

            a = res_a[cfg][level]
            b = res_b[cfg][level]

            logging.info(
                f"{int(level*100):>3}% | "
                f"NSE Δ {a['nse_mean'] - b['nse_mean']:+.3f} | "
                f"MAE Δ {a['mae_mean'] - b['mae_mean']:+.2f}"
            )

# =========================================================
# MODEL FAMILY EXPERIMENT
# =========================================================
def evaluate_model_family(model_type, device, dataframes, scaler_d, configs):

    logging.info(f"\n\n========== MODEL: {model_type} ==========")

    res_no = evaluate_configurations(
        model_type,
        use_physics=False,
        device=device,
        dataframes=dataframes,
        scaler_d=scaler_d,
        configs=configs
    )

    res_phy = evaluate_configurations(
        model_type,
        use_physics=True,
        device=device,
        dataframes=dataframes,
        scaler_d=scaler_d,
        configs=configs
    )

    print_results(res_no,  f"{model_type} No Physics")
    print_results(res_phy, f"{model_type} + Physics")

    compare_results(
        res_phy,
        res_no,
        f"{model_type} Physics",
        f"{model_type} No Physics"
    )

    return {
        "no_physics": res_no,
        "physics": res_phy
    }

# =========================================================
# CROSS MODEL COMPARISON
# =========================================================
def compare_across_models(all_results):

    logging.info("\n\n========== CROSS-MODEL COMPARISON ==========")

    base_model = "MLP"

    for model_type, res in all_results.items():

        if model_type == base_model:
            continue

        logging.info(f"\n--- {model_type} vs {base_model} ---")

        compare_results(
            res["physics"],
            all_results[base_model]["physics"],
            f"{model_type} Physics",
            f"{base_model} Physics"
        )

# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    base = os.path.dirname(__file__)

    logging.info(f"Device: {device}")

    train_df, val_df, test_df, scaler_p, scaler_d, init_rain, init_flood = \
        dataset.load_and_preprocess_data(
            os.path.join(base, "data/01013500_streamflow_qc.txt"),
            os.path.join(base, "data/01013500_lump_cida_forcing_leap.txt")
        )

    dataframes = {
        "train": train_df,
        "val": val_df,
        "test": test_df,
        "thresholds": (init_rain, init_flood)
    }

    configs = [
        {
            "hidden_dim": 128,
            "layer_dim": 2,
            "seq_length": 116,
            "batch_size": 16,
            "lr": 0.0073,
            "lambda_mono": 0.9,
            "lambda_thresh": 0.85,
            "mono_horizon": 3
        }
    ]

    MODEL_ORDER = ["MLP", "RNN", "TCN", "LSTM", "GRU", "Transformer"]

    all_results = {}

    for model_type in MODEL_ORDER:

        all_results[model_type] = evaluate_model_family(
            model_type,
            device,
            dataframes,
            scaler_d,
            configs
        )

    compare_across_models(all_results)

```



# update

**Objective**: to perform model specifc models’ set up and configuration tweaks in order to improve their performance compared the computational weight of the LSTM. Thus, the upgrade should be limitted ot necessesity based, wholestic targeted alterations and upgrades that can be made to the original script while maintaining a scientifically fair comparison between models.

```
# run_uq_tuned6_v11_fixed.py

import torch
import numpy as np
from torch.utils.data import DataLoader
import random
import os
import sys
import copy
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import logging

sys.path.append(os.path.join(os.path.dirname(__file__), 'src'))
import src.utils as utils
import src.models as models
import src.dataset as dataset

# -----------------------------
# Logging
# -----------------------------
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

# -----------------------------
# Seed + Determinism
# -----------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# -----------------------------
# Hydro Prior (fixed usage)
# -----------------------------
class HydroPrior(nn.Module):
    def __init__(self, L):
        super().__init__()
        self.kernel_raw = nn.Parameter(torch.ones(L))
        self.L = L

    def forward(self, r):
        kernel = torch.softmax(self.kernel_raw, dim=0)
        Q = F.conv1d(
            r.transpose(1, 2),
            kernel.view(1, 1, self.L),
            padding=self.L - 1
        )
        return Q[:, 0, :r.size(1)]

# -----------------------------
# Mass balance loss (new)
# -----------------------------

def mass_balance_loss(q, r, extreme, runoff_coeff=0.8, eps=1e-6):
    """
    Event-based mass balance constraint:
    sum(q) <= c * sum(r)

    q: [B, T, 1]
    r: [B, T, 1]
    extreme: rainfall mask
    """

    # mask rainfall periods
    q_masked = q * extreme
    r_masked = r * extreme

    sum_q = q_masked.sum(dim=1)
    sum_r = r_masked.sum(dim=1)

    # upper bound (physical constraint)
    upper_violation = F.relu(sum_q - runoff_coeff * sum_r)

    # optional lower bound (weak, prevents trivial zero flow)
    lower_violation = F.relu(0.1 * sum_r - sum_q)

    # normalize (IMPORTANT)
    norm = sum_r + eps

    loss = (upper_violation / norm) + 0.3 * (lower_violation / norm)

    return loss.mean()

# =========================
# IMPULSE-RESPONSE FORCING
# =========================
def compute_effective_rainfall(r, decay=0.8):
    """
    r: (B, T, 1)
    returns: (B, T, 1) smoothed / lag-aware rainfall
    """
    B, T, _ = r.shape
    S = torch.zeros_like(r)

    for t in range(T):
        if t == 0:
            S[:, t, :] = r[:, t, :]
        else:
            S[:, t, :] = decay * S[:, t-1, :] + r[:, t, :]

    return S

# -----------------------------
# TRAIN (fixed minimal changes)
# -----------------------------
def train_model_v10_adaptive(
    model,
    train_loader,
    val_loader,
    lr,
    use_physics=False,
    use_hydro_prior=False,
    hydro_prior=None,
    use_mass_balance=False,
    epochs=100,
    patience=10,
    warmup_epochs=10,
    device='cpu',
    lambda_mono=0.3,
    lambda_thresh=0.3,
    lambda_hydro=0.1,
    mono_horizon=3,
    lambda_mass=0.05
):

    optimizer = optim.AdamW(model.parameters(), lr=lr)

    if use_hydro_prior and hydro_prior is not None:
        hydro_prior.eval()

    best_val_loss = float('inf')
    best_weights = copy.deepcopy(model.state_dict())
    patience_counter = 0

    for epoch in range(epochs):

        model.train()

        # smoother warmup (avoid sharp physics jump)
        if use_physics:
            alpha = min(1.0, (epoch / warmup_epochs))
        else:
            alpha = 0.0

        for batch in train_loader:
            x = batch['x'].to(device)
            y = batch['y'].to(device)

            optimizer.zero_grad()

            q = model(x)

            # expand target across time dimension
            y_seq = y.unsqueeze(1).expand(-1, q.shape[1], -1)

            # sequence supervision loss
            loss_data = F.mse_loss(q, y_seq)

            loss_phys = torch.tensor(0.0, device=device)

            # =========================
            # PHYSICS LOSS
            # =========================
            if use_physics:
                r = x[:, :, 0].unsqueeze(-1)

                t_rain = model.thresh_rain
                t_flood = model.thresh_flood

                extreme = (r > t_rain).float()

                # ---- monotonicity (impulse + derivative) ----

                # effective rainfall (storage-like smoothing)

                r_eff = compute_effective_rainfall(r, decay=0.8)

                # normalize for stability
                r_eff = r_eff / (r_eff.mean(dim=1, keepdim=True) + 1e-6)

                # ---- align sequence lengths safely ----
                T = min(q.shape[1], r_eff.shape[1])

                q_c = q[:, :T, :]
                r_eff_c = r_eff[:, :T, :]

                # compute derivative AFTER alignment
                forcing = r_eff_c[:, 1:, :]

                # enforce rainfall-response consistency instead of dq dynamics
                q_next = q_c[:, 1:, :]
                q_prev = q_c[:, :-1, :]

                r_influence = forcing  # already aligned

                # softer constraint: response should not drop under sustained forcing
                loss_mono = (F.relu(q_prev - q_next) * r_influence).mean()

                # ---- threshold constraint ----
                t_dyn = t_flood + 0.1 * q[:, :-1, :]

                # align sequence lengths dynamically
                T = min(t_dyn.shape[1], q.shape[1] - 1, extreme.shape[1])
                t_dyn_aligned = t_dyn[:, :T, :]
                q_aligned = q[:, :T, :]
                extreme_aligned = extreme[:, :T, :]

                loss_thresh = (F.relu(t_dyn_aligned - q_aligned) * extreme_aligned).mean()

                loss_phys = (
                    lambda_mono * loss_mono +
                    lambda_thresh * loss_thresh
                )

                # =========================
                # HYDRO PRIOR
                # =========================
                if use_hydro_prior and hydro_prior is not None:
                    Q_prior = hydro_prior(r).unsqueeze(-1).detach()

                    # shape safety
                    T = min(q.shape[1], Q_prior.shape[1], extreme.shape[1])
                    q_c = q[:, :T, :]
                    p_c = Q_prior[:, :T, :]
                    m_c = extreme[:, :T, :]

                    # centered comparison
                    q_c = q_c - q_c.mean(dim=1, keepdim=True)
                    p_c = p_c - p_c.mean(dim=1, keepdim=True)

                    diff2 = (q_c - p_c) ** 2
                    mask_sum = m_c.sum() + 1e-6
                    loss_hydro = (diff2 * m_c).sum() / mask_sum

                    # delayed activation
                    hydro_gate = min(1.0, max(0.0, (epoch - 5) / warmup_epochs))

                    loss_phys = loss_phys + lambda_hydro * hydro_gate * loss_hydro

                # =========================
                # MASS BALANCE
                # =========================
                if use_mass_balance:
                    loss_mass = mass_balance_loss(q, r, extreme)

                    # delayed activation
                    mass_gate = min(1.0, max(0.0, (epoch - 3) / warmup_epochs))

                    loss_phys = loss_phys + lambda_mass * mass_gate * loss_mass

            # =========================
            # TOTAL LOSS
            # =========================
            loss = loss_data + alpha * loss_phys

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        # =========================
        # VALIDATION
        # =========================
        model.eval()
        val_losses = []

        with torch.no_grad():
            for batch in val_loader:
                x = batch['x'].to(device)
                y = batch['y'].to(device)

                pred = model(x)[:, -1, :]
                val_losses.append(F.mse_loss(pred, y).item())

        val_loss = np.mean(val_losses)

        logging.info(
            f"Epoch {epoch+1} | Val Loss: {val_loss:.4f} | Alpha: {alpha:.3f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    model.load_state_dict(best_weights)
    return model

# =========================================================
# LOGGING (MUST BE FIRST)
# =========================================================
os.makedirs("results", exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler("results/run.log"),
        logging.StreamHandler()
    ]
)

# =========================================================
# CONFIGURATIONS (FROM OPTUNA, FIXED)
# =========================================================
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def compute_flops(model, seq_length, hidden_dim, layer_dim):
    """
    Estimate the number of FLOPs for a given model type and configuration.
    """
    if isinstance(model, nn.LSTM):
        flops = 4 * hidden_dim * hidden_dim * seq_length * layer_dim  # LSTM
    elif isinstance(model, nn.GRU):
        flops = 3 * hidden_dim * hidden_dim * seq_length * layer_dim  # GRU
    elif isinstance(model, nn.Linear):
        flops = hidden_dim * seq_length * hidden_dim * layer_dim  # Fully connected
    elif isinstance(model, nn.Transformer):
        # Transformer (multi-head attention and feed-forward)
        flops = 2 * seq_length * hidden_dim * seq_length * layer_dim  # Attention part
        flops += 4 * seq_length * hidden_dim * hidden_dim * layer_dim  # Feed-forward part (approx)
    else:
        flops = hidden_dim * seq_length * hidden_dim * layer_dim  # Default MLP or custom model

    return flops

def match_hidden_dim_by_flops(model_class, cfg, init_rain, init_flood, target_flops, device, model_type=None):

    best_h = cfg["hidden_dim"]
    best_diff = float("inf")
    seq_length = cfg["seq_length"]

    while True:
        # Calculate the number of FLOPs
        flops = compute_flops(model_class, seq_length, best_h, cfg["layer_dim"])

        # Compute relative difference
        diff = abs(flops - target_flops) / target_flops

        # If the difference is small enough, break the loop
        if diff < 0.05:
            break

        # Adjust hidden_dim based on FLOPs
        if flops > target_flops:
            best_h = max(16, best_h - 16)  # Ensure we don't go below 16
        else:
            best_h += 16

    logging.info(f"Best match for FLOPs: hidden_dim={best_h}, FLOPs={flops}")
    return best_h

def get_target_flops(cfg, init_rain, init_flood, device, reference="LSTM"):
    """
    Get the target number of FLOPs based on the reference model (LSTM, GRU, or Transformer).
    
    Args:
    - cfg (dict): Configuration dictionary containing model parameters.
    - init_rain (float): Initial rain threshold.
    - init_flood (float): Initial flood threshold.
    - device (torch.device): The device (CPU or GPU) for model training.
    - reference (str): Reference model to use for computing target FLOPs. (default: "LSTM")
    
    Returns:
    - target_flops (int): The target number of FLOPs for the specified model.
    """
    
    # Define FLOP calculation for each model type based on architecture
    if reference == "LSTM":
        # LSTM: 4 * hidden_dim * hidden_dim * seq_length * layer_dim (input-output & gate computations)
        target_flops = 4 * cfg["hidden_dim"] * cfg["hidden_dim"] * cfg["seq_length"] * cfg["layer_dim"]
    elif reference == "GRU":
        # GRU: 3 * hidden_dim * hidden_dim * seq_length * layer_dim (input-output & gate computations)
        target_flops = 3 * cfg["hidden_dim"] * cfg["hidden_dim"] * cfg["seq_length"] * cfg["layer_dim"]
    elif reference == "Transformer":
        # Transformer: Attention part (2 * seq_length * hidden_dim * seq_length * layer_dim) +
        # Feed-forward part (4 * seq_length * hidden_dim * hidden_dim * layer_dim)
        target_flops = (2 * cfg["seq_length"] * cfg["hidden_dim"] * cfg["seq_length"] * cfg["layer_dim"]) + \
                       (4 * cfg["seq_length"] * cfg["hidden_dim"] * cfg["hidden_dim"] * cfg["layer_dim"])
    else:
        # Default MLP or other architecture
        target_flops = cfg["hidden_dim"] * cfg["seq_length"] * cfg["hidden_dim"] * cfg["layer_dim"]
    
    logging.info(f"[Reference: {reference}] Target FLOPs = {target_flops}")
    return target_flops

# =========================================================
# MODEL BUILDING
# =========================================================
def build_model_matched_by_flops(model_type, cfg, init_rain, init_flood, device, target_flops):

    model_map = {
        "MLP": models.PhysicsInformedMLP,
        "RNN": models.PhysicsInformedRNN,
        "TCN": models.PhysicsInformedTCN,
        "LSTM": models.PhysicsInformedLSTM,
        "GRU": models.PhysicsInformedGRU,
        "Transformer": models.PhysicsInformedTransformer,
    }

    model_class = model_map[model_type]

    hidden_dim = match_hidden_dim_by_flops(
        model_class,
        cfg,
        init_rain,
        init_flood,
        target_flops,
        device,
        model_type=model_type
    )

    model = model_class(
        input_dim=2,
        hidden_dim=hidden_dim,
        layer_dim=cfg["layer_dim"],
        output_dim=1,
        init_rain=init_rain,
        init_flood=init_flood,
        init_weight=0.0
    ).to(device)

    logging.info(
        f"{model_type} | hidden_dim={hidden_dim} | params={count_parameters(model)} | target_flops={target_flops}"
    )

    return model
# =========================================================
# RUN EXPERIMENT (FIXED SIGNATURE)
# =========================================================
def run_experiment_by_flops(cfg, model_type, use_physics, device, dataframes, use_hydro_prior=False, use_mass_balance=False):
    set_seed(42)

    init_rain, init_flood = dataframes["thresholds"]

    # Data loaders
    train_loader = DataLoader(
        dataset.CamelsDataset(dataframes["train"], cfg["seq_length"], True),
        batch_size=cfg["batch_size"],
        shuffle=True,
        num_workers=0  # <- critical for determinism
    )
    val_loader = DataLoader(
        dataset.CamelsDataset(dataframes["val"], cfg["seq_length"], False),
        batch_size=cfg["batch_size"],
        num_workers=0
    )
    test_loader = DataLoader(
        dataset.CamelsDataset(dataframes["test"], cfg["seq_length"], False),
        batch_size=cfg["batch_size"],
        num_workers=0
    )

    # Get target FLOPs for reference model (LSTM by default)
    target_flops = get_target_flops(cfg, init_rain, init_flood, device, reference="LSTM")

    # Build the model based on the FLOP target
    model = build_model_matched_by_flops(
        model_type,
        cfg,
        init_rain,
        init_flood,
        device,
        target_flops
    )

    hydro_prior = None  # removed (not used)

    # Train the model
    model = train_model_v10_adaptive(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        lr=cfg["lr"],
        use_physics=use_physics,
        use_hydro_prior=False,
        hydro_prior=hydro_prior,
        epochs=100,
        patience=12,
        warmup_epochs=7,
        device=device,
        lambda_mono=cfg.get("lambda_mono", 0.5),
        lambda_thresh=cfg.get("lambda_thresh", 0.5),
        mono_horizon=cfg.get("mono_horizon", 3)
    )

    return model, test_loader

# =========================================================
# EVALUATION FUNCTION
# =========================================================
def evaluate(model, loader, device, scaler_d, level):

    preds, targets = utils.evaluate_stress_test(
        model, loader, scaler_d,
        mask_ratio=level,
        device=device
    )

    nse = utils.calculate_nse(targets, preds)

    thr = np.percentile(targets, 90)
    mask = targets > thr
    mae = np.mean(np.abs(preds[mask] - targets[mask])) if np.any(mask) else 0.0

    logging.info(f"Evaluation results: NSE={nse:.3f}, MAE={mae:.2f}")
    return nse, mae

# =========================================================
# CONFIG LOOP
# =========================================================
SCARCITY_LEVELS = [0.0, 0.2, 0.5, 0.8]
M_BOOTSTRAPS = 30

def evaluate_configurations(
    model_type,
    use_physics,
    device,
    dataframes,
    scaler_d,
    configs
):

    results = {}

    for i, cfg in enumerate(configs):

        logging.info(
            f"\n===== Config {i+1} | {model_type} | Physics={use_physics} ====="
        )

        # Use the updated `run_experiment_by_flops` function
        model, test_loader = run_experiment_by_flops(
            cfg=cfg,
            model_type=model_type,
            use_physics=use_physics,
            device=device,
            dataframes=dataframes
        )

        results[f"Config_{i+1}"] = {}

        for level in SCARCITY_LEVELS:

            runs = 1 if level == 0 else M_BOOTSTRAPS
            nses, maes = [], []

            for _ in range(runs):
                nse, mae = evaluate(model, test_loader, device, scaler_d, level)
                nses.append(nse)
                maes.append(mae)

            results[f"Config_{i+1}"][level] = {
                "nse_mean": np.mean(nses),
                "nse_std": np.std(nses),
                "mae_mean": np.mean(maes),
                "mae_std": np.std(maes),
            }

    return results

# =========================================================
# PRINTING (UNCHANGED STYLE)
# =========================================================
def print_results(results, title):

    logging.info(f"\n========== {title} ==========")

    for cfg, levels in results.items():

        logging.info(f"\n🔧 {cfg}")

        for level in sorted(levels.keys()):

            m = levels[level]

            logging.info(
                f"{int(level*100):>3}% | "
                f"NSE {m['nse_mean']:.3f} ± {m['nse_std']:.3f} | "
                f"MAE {m['mae_mean']:.2f} ± {m['mae_std']:.2f}"
            )

# =========================================================
# COMPARISON (YOUR ORIGINAL NICE FORMAT KEPT)
# =========================================================
def compare_results(res_a, res_b, title_a="A", title_b="B"):

    logging.info(f"\n========== COMPARISON: {title_a} vs {title_b} ==========")

    for cfg in res_a.keys():

        logging.info(f"\n🔧 {cfg}")

        for level in sorted(res_a[cfg].keys()):

            a = res_a[cfg][level]
            b = res_b[cfg][level]

            logging.info(
                f"{int(level*100):>3}% | "
                f"NSE Δ {a['nse_mean'] - b['nse_mean']:+.3f} | "
                f"MAE Δ {a['mae_mean'] - b['mae_mean']:+.2f}"
            )

# =========================================================
# MODEL FAMILY EXPERIMENT
# =========================================================
def evaluate_model_family(model_type, device, dataframes, scaler_d, configs):

    logging.info(f"\n\n========== MODEL: {model_type} ==========")

    res_no = evaluate_configurations(
        model_type,
        use_physics=False,
        device=device,
        dataframes=dataframes,
        scaler_d=scaler_d,
        configs=configs
    )

    res_phy = evaluate_configurations(
        model_type,
        use_physics=True,
        device=device,
        dataframes=dataframes,
        scaler_d=scaler_d,
        configs=configs
    )

    print_results(res_no,  f"{model_type} No Physics")
    print_results(res_phy, f"{model_type} + Physics")

    compare_results(
        res_phy,
        res_no,
        f"{model_type} Physics",
        f"{model_type} No Physics"
    )

    return {
        "no_physics": res_no,
        "physics": res_phy
    }

# =========================================================
# CROSS MODEL COMPARISON
# =========================================================
def compare_across_models(all_results):

    logging.info("\n\n========== CROSS-MODEL COMPARISON ==========")

    base_model = "MLP"

    for model_type, res in all_results.items():

        if model_type == base_model:
            continue

        logging.info(f"\n--- {model_type} vs {base_model} ---")

        compare_results(
            res["physics"],
            all_results[base_model]["physics"],
            f"{model_type} Physics",
            f"{base_model} Physics"
        )

# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    base = os.path.dirname(__file__)

    logging.info(f"Device: {device}")

    train_df, val_df, test_df, scaler_p, scaler_d, init_rain, init_flood = \
        dataset.load_and_preprocess_data(
            os.path.join(base, "data/01013500_streamflow_qc.txt"),
            os.path.join(base, "data/01013500_lump_cida_forcing_leap.txt")
        )

    dataframes = {
        "train": train_df,
        "val": val_df,
        "test": test_df,
        "thresholds": (init_rain, init_flood)
    }

    configs = [
        {
            "hidden_dim": 128,
            "layer_dim": 2,
            "seq_length": 116,
            "batch_size": 16,
            "lr": 0.0073,
            "lambda_mono": 0.9,
            "lambda_thresh": 0.85,
            "mono_horizon": 3
        }
    ]

    MODEL_ORDER = ["MLP", "RNN", "TCN", "LSTM", "GRU", "Transformer"]

    all_results = {}

    for model_type in MODEL_ORDER:

        all_results[model_type] = evaluate_model_family(
            model_type,
            device,
            dataframes,
            scaler_d,
            configs
        )

    compare_across_models(all_results)

```



## Evaluation of the Update

**Assessment against the Objective:**

The objective was to make targeted alterations to ensure a scientifically fair comparison between models based on their computational weight relative to the LSTM model.

**Changes made in the updated script:**
1.  **Shift from Parameter Matching to FLOP Matching:** The original script matched models based on the number of trainable parameters (`count_parameters`). The updated script introduces `compute_flops`, `get_target_flops`, and `match_hidden_dim_by_flops`.
2.  **Architectural Awareness:** The `compute_flops` function correctly accounts for the different computational complexities of LSTMs, GRUs, MLPs, and Transformers (e.g., differentiating between the attention mechanism and feed-forward layers in Transformers).
3.  **Dynamic Hidden Dimension Adjustment:** `match_hidden_dim_by_flops` iteratively adjusts the hidden dimension of the candidate model until its FLOP count closely aligns (within 5%) with the target LSTM's FLOP count.

**Conclusion:**
The upgrade is entirely in line with and satisfies the stated objective. Comparing models by FLOPs (Floating Point Operations) is a much more robust measure of computational weight than parameter count, especially when comparing recurrent architectures (LSTMs, GRUs) with non-recurrent ones (MLPs) or attention-based ones (Transformers). This ensures a truly fair scientific comparison where every model is given a similar computational budget.

still, not satisfied with the upgrade. to meet the objective while making limitted and sergical alteration to the original code, the following suggestions were made. check if they make sense:


For a **scientific and fair approach**, each model type may require specific treatments while also within the boundary of similar computational weight. Here’s a recommended **model-specific configuration strategy** for tuning:

### 1. **MLP (Multi-Layer Perceptron)**

* **Focus**: Deepening the architecture by increasing the **number of hidden layers** and/or **hidden units** (`layer_dim`, `hidden_dim`).
* **Activation**: Experiment with **LeakyReLU** or **GELU** instead of ReLU for smoother gradients.
* **Regularization**: **Add dropout** to avoid overfitting, especially for larger models.
* **Initialization**: Use **Xavier** initialization for weights.

### 2. **RNN (Recurrent Neural Network)**

* **Focus**: RNNs benefit from **increased depth** (`layer_dim`) to capture sequential patterns.
* **Hidden State Size**: Increase the **hidden state dimension** (`hidden_dim`) for larger memory capacity.
* **Regularization**: Apply **dropout** between layers in the RNN and after the output layer.
* **Gradient Clipping**: Apply **gradient clipping** to avoid exploding gradients, especially with longer sequences.
* **Initialization**: **Xavier** initialization works well for RNNs.

### 3. **TCN (Temporal Convolutional Network)**

* **Focus**: Increase **depth** by adding more layers or use larger **kernel sizes**.
* **Dilated Convolutions**: Experiment with **larger dilation rates** to capture wider temporal dependencies.
* **Activation**: Switch to **LeakyReLU** or **GELU** for better stability and performance in deeper networks.
* **Regularization**: Apply **dropout** after each convolution layer.

### 4. **LSTM (Long Short-Term Memory)**

* **Focus**: LSTMs often require a balance between **hidden state size** and **depth**. Test **layer_dim** values and increase **hidden_dim** for better performance.
* **Dropout**: Use **dropout** on the **LSTM layers** (via `dropout=0.3`).
* **Gradient Clipping**: Critical for LSTM networks to prevent exploding gradients in long sequences.
* **Initialization**: **Xavier** initialization works here too, but **orthogonal initialization** for LSTM weights may improve performance.

### 5. **GRU (Gated Recurrent Unit)**

* **Focus**: Similar to LSTMs, but often requires **less depth** for stable learning. Experiment with **hidden_dim** to find optimal performance.
* **Dropout**: Use **dropout** inside the GRU layers to avoid overfitting.
* **Gradient Clipping**: Also useful for GRU to avoid gradient explosion.
* **Initialization**: **Xavier** initialization works well, but you can experiment with **orthogonal** initialization if needed.

### 6. **Transformer**

* **Focus**: Experiment with **layer_dim** (number of transformer layers) and **nhead** (number of attention heads). **Increase hidden_dim** for larger models to capture more complex dependencies.
* **Positional Encoding**: For long sequences, you can try **learned positional encodings** instead of sinusoidal.
* **Dropout**: Use **dropout** after attention layers to prevent overfitting.
* **Attention Mechanism**: Test **multi-head attention** by increasing `nhead` for improved performance in large datasets.
* **Activation**: Replace ReLU with **GELU** for smoother learning in attention-based models.


### 7. **General Tips Across All Models**

* **Regularization**: Dropout (`0.2 - 0.5`) helps with overfitting for deeper models.
* **Learning Rate Scheduler**: Use **ReduceLROnPlateau** for dynamic adjustments based on validation loss.
* **Gradient Clipping**: Essential for RNNs, LSTMs, and GRUs.

## Evaluation of Model-Specific Tuning Suggestions

**Assessment against the Objective:**

The stated objective emphasizes **"limited and surgical alterations"** to maintain a **"scientifically fair comparison"** between models while matching their computational weight to the LSTM.

While the suggestions provided are excellent **general best practices** for training deep learning models, they largely conflict with the specific constraints of this experiment. Here is a breakdown of why:

### 1. The Risk of Confounding Variables
If you apply different optimization tricks to different models (e.g., giving the MLP a GELU activation, using Orthogonal initialization for the LSTM, and applying different dropout rates), you introduce confounding variables. If the MLP outperforms the LSTM, it will be impossible to tell if the improvement is due to the feed-forward architecture itself or simply because GELU and Xavier initialization provided a better optimization landscape than the LSTM's setup.

### 2. Conflict with FLOP Matching
Suggestions like *"Deepening the architecture by increasing the number of hidden layers"* or *"Increase the hidden state dimension"* manually contradict the recently implemented `match_hidden_dim_by_flops` function. The FLOP matching algorithm should be the sole mechanism dictating `hidden_dim` and `layer_dim` to ensure strict computational fairness.

### 3. Violation of "Limited and Surgical" Constraint
Implementing custom initialization schemes, different activation functions per model, and unique regularization strategies requires significant rewrites of the `src/models.py` architectures and the training loop, violating the constraint to keep alterations minimal.

---

### Recommended Approach for a Fair Comparison
To integrate these ideas while maintaining scientific rigor, you should apply a **uniform training harness** across all models wherever physically possible:

1. **Global Regularization:** Apply a standard Dropout rate (e.g., `0.2`) uniformly to all models.
2. **Global Gradient Clipping:** Apply gradient clipping (`torch.nn.utils.clip_grad_norm_`) uniformly in the training loop (which is actually already present in `train_model_v10_adaptive` with a value of `1.0`).
3. **Global Learning Rate Scheduler:** Implementing a `ReduceLROnPlateau` scheduler is a fantastic idea, provided it is applied to *all* models identically.
4. **Uniform Activations:** If you upgrade ReLU to GELU, do it across all models that use an activation function (MLP, TCN, Transformer) rather than just one.

**Conclusion:** The suggestions are structurally sound for maximizing individual model performance but must be heavily standardized and uniformly applied to preserve the scientific integrity of the cross-model comparison.

the main claim of the scitific article is provided in the title and abstract. roughly it is to show how ml models leverage physics informed modeling in boosting accuracy, especially in data scares condition.

```

\title{Physics-Informed Fallback Mechanisms Bound Predictive Uncertainty During Sensor Failure in Flood Forecasting}

\begin{abstract}
Deep learning models like Long Short-Term Memory (LSTM) networks have revolutionized hydrological modeling but remain fragile in data-scarce environments where continuous streamflow records are unavailable. Standard Recurrent Neural Networks (RNNs) depend heavily on autoregressive inputs; consequently, their predictive skill collapses when sensor failures disrupt the historical data record. To address this reliance, we propose a Physics-Informed LSTM (PI-LSTM) that integrates learnable physical constraints, specifically monotonicity and flood thresholds, directly into the loss function. This architecture enables the model to learn a "physics fallback" mechanism, inferring discharge dynamics from precipitation even when autoregressive signals are lost. Using a 20-year dataset from the CAMELS-US library (1980–1999), we evaluated model robustness across a stress gradient of 0\% to 80\% missing discharge data. While standard LSTMs failed under severe scarcity (NSE dropping to 0.778 $\pm$ 0.030), the PI-LSTM maintained robust performance (NSE 0.862 $\pm$ 0.022). Crucially, the physical constraints reduced prediction error for extreme flood events by 40.1\% (MAE reduction from 1987 $\pm$ 250 cfs to 1192 $\pm$ 130 cfs) when 80\% of data was missing. These results demonstrate that learnable physical regularization effectively substitutes for lost autoregressive history, which ensures hydrological plausibility and prevents model collapse. This approach offers a resilient framework for flood forecasting in ungaged or poorly monitored basins.
\end{abstract}

```


given that, keeping the computational weight in a similar level, as the lstm, i was thinking trying different model setups to maximize the performance of each model on the problem on hand. do you think you have a different recommendation or erlier suggestion suffiece.

## Evaluation: Core Claim vs. Individual Model Tuning

Given the abstract, your paper’s core scientific claim is that **the Physics-Informed (PI) loss function provides a critical fallback mechanism during data scarcity**, preventing model collapse.

Because the star of the paper is the *loss function modification* (the physics constraints) and the *stress test* (0% to 80% missing data), **the earlier recommendation to use a Uniform Training Harness still stands as the superior scientific approach.**

Here is why individually tweaking models to "maximize performance" is actually detrimental to your paper's specific narrative:

### 1. The Focus is the $\Delta$ (Delta), Not Absolute SOTA
Your abstract highlights the relative improvement: standard LSTM drops to 0.778, while PI-LSTM maintains 0.862. The impact of the paper comes from measuring the $\Delta$ (improvement) granted by the physics constraints.

If you heavily customize the architecture of the TCN (e.g., specific dilations, GELU) but leave the LSTM standard, reviewers will question whether the TCN's response to data scarcity is due to the physics loss, or just because it was uniquely optimized. A uniform harness ensures the *only* variable changing is the addition of physics.

### 2. Generalizability is a Stronger Claim
Showing that the physics constraints improve an "out-of-the-box, standard" RNN, GRU, and MLP is a much stronger scientific claim than showing it works on highly customized, surgically tweaked variants. It proves your physics loss is robust and architecture-agnostic.

### 3. The FLOP Constraint is Sufficient for Fairness
By already matching the computational weight (FLOPs) as implemented in the recent upgrade, you have leveled the playing field. An MLP with identical FLOPs to an LSTM is a fair baseline.

### The Exception: Automated Hyperparameter Optimization (HPO)
If you feel you *must* tune the baselines so reviewers don't claim you used "weak baselines," you cannot do it manually.
* **The scientifically sound compromise:** Run an automated search (like Optuna) to find the best standard hyperparameters (Learning Rate, standard Dropout) for each base model type on the 0% scarcity baseline.
* **Freeze those parameters.**
* Then, apply the physics loss and run the 0%-80% scarcity stress test using those frozen parameters.

**Final Verdict:** Do not manually tune activations or initializations per model. Stick to the FLOP-matched, uniformly regularized setup (standard dropout, clipping, scheduling) to keep the spotlight strictly on your novel physics-informed fallback mechanism.

Good. the implemenation of the ugraded script provided undesirable result, such as different performance matrics even for lstm model. could you provide better suggestion to implement of the original script

## Diagnosing the FLOP-Matching Issue

If the LSTM model's performance metrics changed after the upgrade, it means the upgrade violated the principle of being "limited and surgical." The reference model (LSTM) should produce identical results to the original script.

### Why did the LSTM metrics change?
1. **Random Seed Interaction**: The original script had carefully placed `set_seed()` calls. If the loop structure or initialization order changed, the random weights assigned to the LSTM would change, leading to different results.
2. **Reference Model Alteration**: In the upgraded `build_model_matched_by_flops` function, the LSTM itself was likely passed through the `match_hidden_dim_by_flops` function. Even if the logic *should* return the original `hidden_dim`, slight math differences might have caused it to step to a different dimension (e.g., jumping by 16).
3. **Initialization Order**: Generating the "target FLOPs" requires instantiating a reference model or doing math before the actual model is instantiated. If random numbers are drawn during this phase, it shifts the global random state for the actual model initialization.

### The Corrected Implementation Strategy (The "Anchor" Method)

To safely implement computational fairness *without* disrupting the original LSTM baselines, follow these strict rules:

**1. Anchor the LSTM (and GRU):**
Do not calculate FLOPs for the LSTM to determine its own hidden dimension. The LSTM must use the exact `hidden_dim` from the `cfg` dictionary, bypassing any matching logic.
```python
# Inside build_model
if model_type in ["LSTM", "GRU"]:
    hidden_dim = cfg["hidden_dim"] # STRICT ANCHOR
else:
    hidden_dim = match_hidden_dim_by_flops(...)
```

**2. Pure Analytical FLOPs (No Model Instantiation):**
Ensure `compute_flops` is a pure mathematical function that relies *only* on integers (`seq_length`, `hidden_dim`, `layer_dim`) and does not instantiate PyTorch modules. Instantiating modules consumes random numbers from the seed.

**3. Uniform Regularization (As Discussed):**
Instead of changing model internals, add a standard dropout parameter to the base configuration:
```python
configs = [{
    "hidden_dim": 128,
    "layer_dim": 2,
    "dropout": 0.2, # Add this globally
    # ...
}]
```
Pass this dropout rate explicitly to the PyTorch LSTM/GRU/RNN modules and add `nn.Dropout(0.2)` layers to the MLP and TCN definitions.

By anchoring the LSTM securely, you guarantee that the baseline metrics match the original script exactly, while the other models scale up or down gracefully to meet the LSTM's computational footprint.

good suggestions. please write the updated script which will work on the original script as per your suggestions. make sure to mention which functions are additions and which are replacements.

## Updated Script: The Anchor Method

Here is the updated script. It ensures the reference models (LSTM and GRU) are strictly anchored to the configuration provided, guaranteeing their metrics remain unchanged from the original run. The other models scale to match the LSTM's computational weight using pure mathematical FLOP estimation (no module instantiation, preserving random seeds).

### Summary of Changes:
*   **Addition**: `compute_flops` (pure mathematical FLOP estimation).
*   **Addition**: `get_target_flops` (calculates target FLOPs based on the LSTM reference).
*   **Replacement**: `match_hidden_dim` is replaced by `match_hidden_dim_by_flops` (uses FLOPs instead of parameters).
*   **Replacement**: `build_model_matched` (now anchors LSTM/GRU and uses FLOP matching for others).
*   **Replacement**: `run_experiment` (updated to use the new FLOP targeting logic).

```python
# run_uq_tuned6_v12_anchor.py

import torch
import numpy as np
from torch.utils.data import DataLoader
import random
import os
import sys
import copy
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import logging

sys.path.append(os.path.join(os.path.dirname(__file__), 'src'))
import src.utils as utils
import src.models as models
import src.dataset as dataset

# -----------------------------
# Logging
# -----------------------------
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

# -----------------------------
# Seed + Determinism
# -----------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ... [KEEP HydroPrior, mass_balance_loss, compute_effective_rainfall EXACTLY AS ORIGINAL] ...
# ... [KEEP train_model_v10_adaptive EXACTLY AS ORIGINAL] ...

# =========================================================
# FLOP MATCHING UTILITIES (NEW/REPLACEMENTS)
# =========================================================
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def compute_flops(model_type, seq_length, hidden_dim, layer_dim):
    """
    [ADDITION] Pure mathematical estimation of FLOPs to preserve random seeds.
    """
    if model_type == "LSTM":
        return 4 * hidden_dim * hidden_dim * seq_length * layer_dim
    elif model_type == "GRU":
        return 3 * hidden_dim * hidden_dim * seq_length * layer_dim
    elif model_type in ["MLP", "RNN", "TCN"]:
        return hidden_dim * seq_length * hidden_dim * layer_dim
    elif model_type == "Transformer":
        # Attention + Feed-forward approximation
        return (2 * seq_length * hidden_dim * seq_length * layer_dim) + \
               (4 * seq_length * hidden_dim * hidden_dim * layer_dim)
    return hidden_dim * seq_length * hidden_dim * layer_dim

def match_hidden_dim_by_flops(model_type, cfg, target_flops):
    """
    [REPLACEMENT] Adjusts hidden_dim analytically based on FLOPs.
    """
    best_h = cfg["hidden_dim"]
    best_diff = float("inf")
    seq_length = cfg["seq_length"]
    layer_dim = cfg["layer_dim"]

    # Analytical search without instantiating models
    h_min, h_max = 16, 2048
    while h_min <= h_max:
        h_mid = (h_min + h_max) // 2
        
        flops = compute_flops(model_type, seq_length, h_mid, layer_dim)
        diff = abs(flops - target_flops) / target_flops

        if diff < best_diff:
            best_diff = diff
            best_h = h_mid

        if flops > target_flops:
            h_max = h_mid - 1
        else:
            h_min = h_mid + 1

        if diff < 0.01: # 1% tolerance
            break
            
    # Ensure even hidden dimensions for transformer heads if needed
    if model_type == "Transformer" and best_h % 2 != 0:
        best_h += 1

    return best_h

def get_target_flops(cfg, reference="LSTM"):
    """
    [ADDITION] Gets target FLOPs mathematically.
    """
    target_flops = compute_flops(reference, cfg["seq_length"], cfg["hidden_dim"], cfg["layer_dim"])
    logging.info(f"[Reference: {reference}] Target FLOPs = {target_flops}")
    return target_flops

# =========================================================
# MODEL BUILDING
# =========================================================
def build_model_matched(model_type, cfg, init_rain, init_flood, device, target_flops):
    """
    [REPLACEMENT] Anchors LSTM/GRU, matches others by FLOPs.
    """
    model_map = {
        "MLP": models.PhysicsInformedMLP,
        "RNN": models.PhysicsInformedRNN,
        "TCN": models.PhysicsInformedTCN,
        "LSTM": models.PhysicsInformedLSTM,
        "GRU": models.PhysicsInformedGRU,
        "Transformer": models.PhysicsInformedTransformer,
    }

    model_class = model_map[model_type]

    # STRICT ANCHOR: Do not alter hidden_dim for reference models
    if model_type in ["LSTM", "GRU"]:
        hidden_dim = cfg["hidden_dim"]
    else:
        hidden_dim = match_hidden_dim_by_flops(
            model_type,
            cfg,
            target_flops
        )

    model = model_class(
        input_dim=2,
        hidden_dim=hidden_dim,
        layer_dim=cfg["layer_dim"],
        output_dim=1,
        init_rain=init_rain,
        init_flood=init_flood,
        init_weight=0.0
    ).to(device)

    actual_flops = compute_flops(model_type, cfg["seq_length"], hidden_dim, cfg["layer_dim"])
    logging.info(
        f"{model_type} | hidden_dim={hidden_dim} | params={count_parameters(model)} | FLOPs={actual_flops}"
    )

    return model

# =========================================================
# RUN EXPERIMENT
# =========================================================
def run_experiment(cfg, model_type, use_physics, device, dataframes, use_hydro_prior=False, use_mass_balance=False):
    """
    [REPLACEMENT] Updated to use FLOP matching.
    """
    set_seed(42) # Ensure seed is identical to original

    init_rain, init_flood = dataframes["thresholds"]

    train_loader = DataLoader(
        dataset.CamelsDataset(dataframes["train"], cfg["seq_length"], True),
        batch_size=cfg["batch_size"],
        shuffle=True,
        num_workers=0  # <- critical for determinism
    )
    val_loader = DataLoader(
        dataset.CamelsDataset(dataframes["val"], cfg["seq_length"], False),
        batch_size=cfg["batch_size"],
        num_workers=0
    )
    test_loader = DataLoader(
        dataset.CamelsDataset(dataframes["test"], cfg["seq_length"], False),
        batch_size=cfg["batch_size"],
        num_workers=0
    )

    target_flops = get_target_flops(cfg, reference="LSTM")

    model = build_model_matched(
        model_type,
        cfg,
        init_rain,
        init_flood,
        device,
        target_flops
    )

    hydro_prior = None

    model = train_model_v10_adaptive(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        lr=cfg["lr"],
        use_physics=use_physics,
        use_hydro_prior=False,
        hydro_prior=hydro_prior,
        epochs=100,
        patience=12,
        warmup_epochs=7,
        device=device,
        lambda_mono=cfg.get("lambda_mono", 0.5),
        lambda_thresh=cfg.get("lambda_thresh", 0.5),
        mono_horizon=cfg.get("mono_horizon", 3)
    )

    return model, test_loader

# ... [KEEP evaluate, evaluate_configurations, print_results, compare_results, evaluate_model_family, compare_across_models, and main block EXACTLY AS ORIGINAL] ...
```

one more question, the algorithm tries to find only optimum hidden dim. will be more reasonable to finding depth, or numb of head, etc

## Evaluating the Search Space: Hidden Dimension vs. Depth/Heads

Your question is highly relevant. Currently, the algorithm scales models by adjusting only the `hidden_dim`. Is it more reasonable to adjust depth (`layer_dim`) or the number of attention heads (`nhead`)?

Here is a breakdown of the trade-offs and a recommendation for your experimental design:

### 1. Scaling `hidden_dim` (Current Approach)
* **Pros:** Highly granular. FLOPs scale quadratically with `hidden_dim` in most recurrent models, but because it's a continuous-like integer space, you can hit the target FLOP budget with extremely high precision (usually within 1-2%).
* **Cons:** "Wide and shallow" networks might behave differently than "narrow and deep" networks, potentially disadvantaging models that require depth to build hierarchical representations (like deep MLPs or TCNs).

### 2. Scaling `layer_dim` (Depth)
* **Pros:** Depth often correlates strongly with a model's ability to learn complex, abstract representations. A deeper MLP might perform much better than a vastly wider one.
* **Cons:** Highly discrete and chunky. Adding a single layer might increase the FLOP count by 50% or 100%, making it nearly impossible to match the LSTM's FLOP budget precisely. You would end up with models that are vastly over- or under-parameterized compared to the target.

### 3. Scaling `nhead` (Transformers Only)
* **Pros:** Attention heads divide the feature space, allowing the model to attend to different representation subspaces simultaneously.
* **Cons:** The standard Transformer architecture dictates that `hidden_dim` must be divisible by `nhead`. Scaling heads without scaling `hidden_dim` just chops the embedding into smaller, weaker pieces.

### Scientific Recommendation

For the specific goal of this paper—isolating the effect of the **Physics-Informed Loss** under data scarcity—**scaling only the `hidden_dim` is the most scientifically sound approach.**

**Why?**
* **Precision:** It guarantees that the computational budget (FLOPs) is tightly controlled. If you scale depth, the discrete jumps in FLOPs will introduce computational imbalances, giving reviewers a reason to doubt the fairness of the baseline comparisons.
* **Controlled Variables:** Keeping the depth (`layer_dim = 2`) constant across all models removes architecture depth as a confounding variable. You are strictly comparing how different *temporal processing mechanisms* (recurrence, convolution, attention, or none) respond to the physics constraints under the exact same depth and computational budget.

**If you want to explore depth:**
The best way to incorporate depth is not to let the FLOP-matcher alter it dynamically. Instead, add a separate configuration to your `configs` list:
```python
configs = [
    { 'hidden_dim': 128, 'layer_dim': 2, ... }, # Shallow experiment
    { 'hidden_dim': 64,  'layer_dim': 4, ... }  # Deep experiment
]
```
This ensures that for any given run, all models share the same depth and are precisely matched in FLOPs via the `hidden_dim` adjustment.

## Implementation Suggestions: Anchoring LSTM and Exploring the Search Space

### 1. Skipping the LSTM (The "Anchor" Method)
As introduced in the updated script above, skipping the LSTM during the FLOP-matching phase is the most robust way to ensure your baseline remains untouched. Here is the specific logic you should focus on:

```python
# Inside your model building function
def build_model_matched(model_type, cfg, init_rain, init_flood, device, target_flops):
    ...
    # STRICT ANCHOR: Do not alter hidden_dim for reference models
    if model_type in ["LSTM", "GRU"]:
        hidden_dim = cfg["hidden_dim"]
    else:
        # For MLP, TCN, Transformer, etc., find the hidden_dim that matches the LSTM's FLOPs
        hidden_dim = match_hidden_dim_by_flops(model_type, cfg, target_flops)
    ...
```
**Why this is critical:** It guarantees that the LSTM (your primary baseline) uses exactly `128` (or whatever you set in `cfg`) and behaves exactly as it did in your original experiments. Only the *competitor* models flex their width to match the LSTM's computational budget.

### 2. Exploring the Search Space (Depth vs. Width)
To scientifically explore the effect of depth (`layer_dim`) across the variety of models without breaking the FLOP fairness, you should define a grid of configurations rather than dynamically changing depth per model.

Instead of having one config, set up your `configs` list in the `main` block to test different "shapes" of models. For each configuration, the LSTM will define the target FLOPs, and all other models will adjust their `hidden_dim` to match that specific shape's FLOP budget.

```python
# In your __main__ block:
configs = [
    {
        "name": "Standard (Balanced)",
        "hidden_dim": 128,
        "layer_dim": 2,      # Standard depth
        "seq_length": 116,
        "batch_size": 16,
        "lr": 0.0073,
        "dropout": 0.2       # Added for uniform regularization
    },
    {
        "name": "Deep (More Layers, Narrower base)",
        "hidden_dim": 90,    # Reduced to keep roughly similar total FLOPs to config 1
        "layer_dim": 4,      # Deeper architecture
        "seq_length": 116,
        "batch_size": 16,
        "lr": 0.0073,
        "dropout": 0.2
    },
    {
        "name": "Shallow (Fewer Layers, Wider base)",
        "hidden_dim": 180,   # Increased base width
        "layer_dim": 1,      # Very shallow
        "seq_length": 116,
        "batch_size": 16,
        "lr": 0.0073,
        "dropout": 0.2
    }
]
```

**How this works in practice:**
1.  **For the "Deep" config:** The script calculates the FLOPs of an LSTM with `hidden_dim=90` and `layer_dim=4`. Let's say that's `X` FLOPs.
2.  Then, when it evaluates the MLP for the "Deep" config, it keeps `layer_dim=4` and finds the specific MLP `hidden_dim` that equals `X` FLOPs.
3.  This ensures that when you compare the Deep MLP to the Deep LSTM, they both have exactly 4 layers and exactly identical FLOPs.

This approach allows you to rigorously test if certain models (like MLPs or Transformers) perform significantly better when deeper or wider, while completely insulating you from criticism about unfair computational advantages.